## 1. Librerías y cargado del dataset

In [162]:
#Librerías
from google.colab import drive
import pandas as pd
import numpy as np

In [132]:
# Se carga el dataset a través de Drive
drive.mount('/content/drive')

ruta = '/content/drive/MyDrive/10ª Edición Máster en Data Science/TFM/Dataset_sin_tratar.csv'
dataset_original = pd.read_csv(ruta,encoding='Latin-1')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Exploración básica

In [133]:
print(dataset_original.shape)
dataset_original.info()

#Para observar las primera 15/300 características del dataset
list(dataset_original.columns)[:15]

(441456, 330)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441456 entries, 0 to 441455
Columns: 330 entries, _STATE to _AIDTST3
dtypes: float64(324), int64(4), object(2)
memory usage: 1.1+ GB


['_STATE',
 'FMONTH',
 'IDATE',
 'IMONTH',
 'IDAY',
 'IYEAR',
 'DISPCODE',
 'SEQNO',
 '_PSU',
 'CTELENUM',
 'PVTRESD1',
 'COLGHOUS',
 'STATERES',
 'CELLFON3',
 'LADULT']

In [134]:
#Figuran solo 328 características porque 2 de las 330 son "objects"
dataset_original.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
_STATE,441456.0,2.996871e+01,1.603471e+01,1.0,19.0,29.0,44.0,72.0
FMONTH,441456.0,6.359676e+00,3.487131e+00,1.0,3.0,6.0,9.0,12.0
IDATE,441456.0,6.563745e+06,3.488675e+06,1012016.0,3232015.0,6242015.0,10022015.0,12312015.0
IMONTH,441456.0,6.416803e+00,3.492082e+00,1.0,3.0,6.0,10.0,12.0
IDAY,441456.0,1.449273e+01,8.335468e+00,1.0,8.0,14.0,21.0,31.0
...,...,...,...,...,...,...,...,...
_RFSEAT2,441456.0,1.824624e+00,2.360812e+00,1.0,1.0,1.0,1.0,9.0
_RFSEAT3,441456.0,1.887028e+00,2.351387e+00,1.0,1.0,1.0,1.0,9.0
_FLSHOT6,157954.0,2.290705e+00,2.518086e+00,1.0,1.0,1.0,2.0,9.0
_PNEUMO2,157954.0,2.412259e+00,2.778032e+00,1.0,1.0,1.0,2.0,9.0


In [135]:
#La variable targer será DIABETE3, siendo las clases que nos importan: 1 (SI) - 3 (NO) - 4 (PREDIABETES)

conteo_target = dataset_original['DIABETE3'].value_counts()

#El hiperparámetro Normalize devuelve la frecuencia relativa con respecto del total
porcentaje_clases_target = dataset_original['DIABETE3'].value_counts(normalize = True)*100

resultado = pd.DataFrame({
    'Conteo': conteo_target,
    'Porcentaje': porcentaje_clases_target
}).transpose()

#Las 3 clases más numerosas son las que nos interesan, aunque están muy desbalanceadas (posibilidad unificar NO y PREDIABETES, o eliminar esta última)
print(resultado)

DIABETE3              3.0           1.0          4.0          2.0         7.0  \
Conteo      372104.000000  57256.000000  7690.000000  3608.000000  598.000000   
Porcentaje      84.291504     12.970015     1.741991     0.817308    0.135463   

DIABETE3          9.0  
Conteo      193.00000  
Porcentaje    0.04372  


## 3. Limpieza y filtrado

In [136]:
# Se eliminan algunas características que no aportan valor al modelo (razonado en el documento: 1. Cribado de columnas)
columnas_para_eliminar = '''
_STATE, FMONTH, IDATE, IMONTH, IDAY, IYEAR, DISPCODE, SEQNO, _PSU, QSTVER, QSTLANG, MSCODE, _STSTR, _STRWT, _RAWRAKE, _WT2RAKE, _CLLCPWT, _DUALUSE, _DUALCOR, _LLCPWT, CTELENUM, CELLFON3,
CTELNUM1, CELLFON2, CADULT, CCLGHOUS, CSTATE, LANDLINE, NUMPHON2, CPDEMO1, INTERNET, PVTRESD1, COLGHOUS, STATERES, LADULT, NUMADULT, NUMMEN, NUMWOMEN, PVTRESD2, HHADULT, HLTHPLN1, PERSDOC2,
MEDCOST, CHECKUP1, RENTHOM1, NUMHHOL2, CHILDREN, INCOME2, CDHELP, CDSOCIAL, _CHLDCNT, _INCOMG, DIABAGE2, CVDINFR4, CVDCRHD4, CVDSTRK3, CHCKIDNY, PDIABTST, PREDIAB1, INSULIN, FEETCHK2,
DOCTDIAB, CHKHEMO3, FEETCHK, EYEEXAM, DIABEYE, DIABEDU, MARITAL, EDUCA, VETERAN3, EMPLOY1, SEATBELT, CDDISCUS, SCNTMNY1, SCNTMEL1, SCNTPAID, SCNTWRK1, SCNTLPAD, SCNTLWK1, SXORIENT, TRNSGNDR,
RCSGENDR, RCSRLTN2, EMTSUPRT, LSATISFY, ADPLEASR, ADDOWN, MISTMNT, _EDUCAG, _RFSEAT2, _RFSEAT3, _LMTWRK1, _LMTSCL1, CAREGIV1, CRGVREL1, CRGVLNG1, CRGVHRS1, CRGVPRB1, CRGVPERS, CRGVHOUS,
CRGVMST2, CRGVEXPT, CDHOUSE, CDASSIST, WEIGHT2, HEIGHT3, HTIN4, HTM4, WTKG3, _BMI5, _RFBMI5, STOPSMK2, LASTSMK2, USENOW3, _SMOKER3, _RFSMOK3, ALCDAY5, DRNK3GE5, MAXDRNKS, DRNKANY5, DROCDY3_,
_RFBING5, _DRNKWEK, _RFDRHV5, FVGREEN, ARTHEXER, FVORANG, VEGETAB1, GRENDAY_, ORNGDAY_, VEGEDA1_, _MISVEGN, _VEGLT1, _VEG23, _VEGETEX, FTJUDA1_, FRUTDA1_, FRUITJU1, FRUIT1, _MISFRTN, _FRTLT1, _FRT16,
_FRUITEX, FVBEANS, EXERANY2, EXRACT11, EXRACT21, EXEROFT2, EXERHMM2, ADMOVE, EXACTOT1, EXACTOT2, _TOTINDA, METVL11_, METVL21_, MAXVO2_, FC60_, ACTIN11_, ACTIN21_, PADUR1_, PADUR2_, PAFREQ1_,
PAFREQ2_, _MINAC11, _MINAC21, STRFREQ_, PAMIN11_, PAMIN21_, PA1MIN_, PAVIG11_, PAVIG21_, PA1VIGM_, _PA150R2, _PA300R2, _PA30021, _PASTRNG, _PAREC1, _PASTAE1, _LMTACT1, LMTJOIN3, ARTHSOCL,
JOINPAIN, PAINACT2, TOLDHI2, QLMENTL2, QLSTRES2, QLHLTH2, _RFHLTH, VIDFCLT2, VIREDIF3, VIPRFVS2, VINOCRE2, VIEYEXM2, VIINSUR2, VICTRCT4, VIGLUMA2, VIMACDG2, ASTHMAGE, ASATTACK, ASERVIST,
ASDRVIST, ASRCHKUP, ASACTLIM, ASYMPTOM, ASNOSLEP, ASTHMED3, ASINHALR, CASTHDX2, CASTHNO2, _LTASTH1, _CASTHM1, _ASTHMS1, HAREHAB1, STREHAB1, CVDASPRN, ASPUNSAF, _MICHD, RLIVPAIN, RDUCHART,
RDUCSTRK, ARTTODAY, ARTHWGT, ARTHEDU, _DRDXAR1, _RFCHOL,  _CHISPNC, _CRACE1, _CPRACE, _PRACE1, _MRACE1, _HISPANC, _RACEG21, _RACEGR3, _RACE_G1, _AGE65YR, _AGE80, _AGE_G, _RFHYPE5,
CHOLCHK, HIVTST6, HIVTSTD3, WHRTST10, HADMAM, HOWLONG, HADPAP2, LASTPAP2, HPVTEST, HPLSTTST, HADHYST2, PROFEXAM, LENGEXAM, BLDSTOOL, LSTBLDS3, HADSIGM3, HADSGCO1, LASTSIG3, PSATEST1, PSATIME,
PCPSARS1, PCPSADE1, PCDMDECN, _AIDTST3, _CHOLCHK, FLUSHOT6, FLSHTMY2, IMFVPLAC, PNEUVAC3, TETANUS, HPVADVC2, HPVADSHT, SHINGLE2, _FLSHOT6, _PNEUMO2, DRADVISE, PREGNANT, PCPSAAD2, PCPSADI1,
PCPSARE1, _HCVU651, STRENGTH, PAMISS1_, SMOKDAY2, EXERHMM1, WTCHSALT, LONGWTCH, BEANDAY_, _FRTRESP, _VEGRESP, _PAINDX1, CIMEMLOS, ASTHMA3, ASTHNOW, CHCSCNCR, CHCOCNCR, CHCCOPD1, BLDSUGAR
'''
# Se eliminan los saltos de línea
columnas_para_eliminar = columnas_para_eliminar.replace("\n", "")
# Se eliminan espacios que pueda haber en las variables
columnas_para_eliminar = columnas_para_eliminar.replace(" ", "")
# Se obtiene una lista a partir de una string
columnas_eliminar = columnas_para_eliminar.split(',')

# De las 330 columnas, se eliminarán 296 por lo que se tendría que tener un nuevo df de 34 columnas (33 variables + 1 target)
print(f'Número de columnas a eliminar: {len(columnas_eliminar)}')
# Se comprueba que no hay ninguna columna duplicada
print(f'Columnas a eliminar únicas: {len(set(columnas_eliminar))}')

#Primero se crea una compia del df original
dataset_pretratrado = dataset_original

#Se comprueba que todas las columnas existen en el df y se incluye una que no existe "prueba" para probar el código
columnas_no_existentes = [col for col in columnas_eliminar if col not in dataset_pretratrado.columns]

if columnas_no_existentes:
    print("Las siguientes columnas NO existen en el DataFrame:")
    for col in columnas_no_existentes:
        print(col)
else:
    print('Todas las columnas existen en el DataFrame')

#Se eliminan las columnas de la lista.
#Se se incluye errors='ignore' se evita que si una columna está mal escrita, haya error, aunque se comprobó en el paso previo.
dataset_pretratrado = dataset_pretratrado.drop(columns=columnas_eliminar, errors='ignore')

Número de columnas a eliminar: 296
Columnas a eliminar únicas: 296
Todas las columnas existen en el DataFrame


In [137]:
#Se comprueba que se tienen 33 variables y 1 target
dataset_pretratrado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441456 entries, 0 to 441455
Data columns (total 34 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   441454 non-null  float64
 1   PHYSHLTH  441455 non-null  float64
 2   MENTHLTH  441456 non-null  float64
 3   POORHLTH  226964 non-null  float64
 4   BPHIGH4   441455 non-null  float64
 5   BPMEDS    178188 non-null  float64
 6   BLOODCHO  441456 non-null  float64
 7   HAVARTH3  441455 non-null  float64
 8   ADDEPEV2  441456 non-null  float64
 9   DIABETE3  441449 non-null  float64
 10  SEX       441456 non-null  float64
 11  QLACTLM2  432118 non-null  float64
 12  USEEQUIP  431026 non-null  float64
 13  BLIND     430302 non-null  float64
 14  DECIDE    429716 non-null  float64
 15  DIFFWALK  429122 non-null  float64
 16  DIFFDRES  428728 non-null  float64
 17  DIFFALON  428130 non-null  float64
 18  SMOKE100  427201 non-null  float64
 19  AVEDRNK2  210838 non-null  float64
 20  EXER

In [138]:
#Se filtran las observaciones que tienen clases del target que se necesitan (3, 1) con 2 clases
lista_targets = [1, 3]
dataset_pretratado = dataset_pretratrado[dataset_pretratrado['DIABETE3'].isin(lista_targets)].copy()

#Se comprobó antes que tendría que haber 437.050  registros. El índice se reseteará cuando se termine de filtrar
dataset_pretratado['DIABETE3'].info()

<class 'pandas.core.series.Series'>
Index: 429360 entries, 0 to 441455
Series name: DIABETE3
Non-Null Count   Dtype  
--------------   -----  
429360 non-null  float64
dtypes: float64(1)
memory usage: 6.6 MB


In [139]:
#Se comprueban cuales son las columnas que tienen mayor número de nulos
nulls = dataset_pretratado.isnull().sum().sort_values(ascending=False)
nulls_pct = (dataset_pretratado.isnull().sum()/len(dataset_pretratado)*100).sort_values(ascending=False)
pd.DataFrame({"Nulos": nulls, "Porcentaje": nulls_pct})

,Nulos,Porcentaje
ADANXEV,409608,95.399665
ADTHINK,409571,95.391047
ADFAIL,409557,95.387786
ADEAT1,409550,95.386156
ADENERGY,409539,95.383594
ADSLEEP,409534,95.382430
ARTHDIS2,297604,69.313397
BPMEDS,257357,59.939678
AVEDRNK2,223367,52.023244
POORHLTH,209652,48.828955


In [140]:
# De las variables preseleccionadas, se eliminan las que mayor número de NaN presentan (razonado en el documento: 1.1 Segundo cribado de columnas)
columnas_eliminar_2 = ['ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL', 'ADTHINK', 'ADANXEV', 'ARTHDIS2']

# Se valora conservar las columnas POORHLTH y AVEDRNK2. En este conteo, no se tienen en cuenta los NaN
porcentaje = dataset_pretratado['POORHLTH'].value_counts(normalize=True)*100
print(porcentaje.head(5))

porcentaje2 = dataset_pretratrado['AVEDRNK2'].value_counts(normalize=True)*100
print(porcentaje2.head(5))

# Lista actualizada:
columnas_eliminar_2 = ['ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL', 'ADTHINK', 'ADANXEV', 'ARTHDIS2', 'POORHLTH', 'AVEDRNK2']

dataset_pretratado_2 = dataset_pretratado.drop(columns=columnas_eliminar_2, errors='ignore')

POORHLTH
88.0    55.651592
30.0     8.620988
2.0      5.436306
1.0      4.841426
5.0      3.572924
Name: proportion, dtype: float64
AVEDRNK2
1.0    46.277237
2.0    29.561559
3.0    10.741897
4.0     4.502509
5.0     2.368169
Name: proportion, dtype: float64


In [141]:
#Se comprueba que se tienen 24 variables y 1 target
dataset_pretratado_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 429360 entries, 0 to 441455
Data columns (total 25 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   429358 non-null  float64
 1   PHYSHLTH  429359 non-null  float64
 2   MENTHLTH  429360 non-null  float64
 3   BPHIGH4   429359 non-null  float64
 4   BPMEDS    172003 non-null  float64
 5   BLOODCHO  429360 non-null  float64
 6   HAVARTH3  429359 non-null  float64
 7   ADDEPEV2  429360 non-null  float64
 8   DIABETE3  429360 non-null  float64
 9   SEX       429360 non-null  float64
 10  QLACTLM2  420246 non-null  float64
 11  USEEQUIP  419180 non-null  float64
 12  BLIND     418473 non-null  float64
 13  DECIDE    417904 non-null  float64
 14  DIFFWALK  417330 non-null  float64
 15  DIFFDRES  416945 non-null  float64
 16  DIFFALON  416362 non-null  float64
 17  SMOKE100  415457 non-null  float64
 18  EXEROFT1  286457 non-null  float64
 19  _RACE     429360 non-null  float64
 20  _AGEG5YR 

In [142]:
# Se crea una clase para visualizar el % de los datos faltantes en las columnas elegidas que no son NaN
class PorcentajesValoresFaltantes:
  def __init__(self, df):
    self.df = df

  # Se calcula el porcentaje, por columna en función de los valores de la lista que representan valores faltantes
  def calcular_porcentaje(self, columnas, valores_especiales):
    resultados = {'Columna': [], 'Porcentaje': []}

    for col in columnas:
      total = self.df[col].notna().sum()
      contar = self.df[col].isin(valores_especiales).sum()
      porcentaje = (contar / total) *100
      resultados['Columna'].append(col)
      resultados['Porcentaje'].append(porcentaje)

    return pd.DataFrame(resultados)

  # Grupos_columnas (diccionario):
  # Clave es el nombre del grupo / Valor es una tupla (columnas, valores_especiales)
  def calcular_todos(self, grupo_columnas):
    resultados_globales = {}

    for nombre, (cols, vals) in grupo_columnas.items():
      resultados_globales[nombre] = self.calcular_porcentaje(cols, vals)

    return resultados_globales


In [143]:
# Se crea el objeto de la clase PorcentajesValoresFaltantes con el df
analizador = PorcentajesValoresFaltantes(dataset_pretratado_2)

# Se crea el diccionario con nombre del grupo / columnas y valores especiales
grupos_columnas = {
    '7_9': (['GENHLTH', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_PACAT1'], [7, 9]),
    '77_88_99': (['PHYSHLTH', 'MENTHLTH'], [77, 88, 99]),
    '777_999': (['EXEROFT1'], [777, 999]),
    '9': (['_RACE'], [9]),
    '14': (['_AGEG5YR'], [14])
}

# Se crea la variable resultados con el metodo calcular_todos del objeto (instancia) analizador
resultados = analizador.calcular_todos(grupos_columnas)

# Se muestan los resultados individuales
for nombre, df_resultado in resultados.items():
    print(f"\n=== Resultados para grupo {nombre} ===")
    print(df_resultado)


=== Resultados para grupo 7_9 ===
     Columna  Porcentaje
0    GENHLTH    0.277158
1    BPHIGH4    0.285542
2     BPMEDS    0.171509
3   BLOODCHO    2.138066
4   HAVARTH3    0.587853
5   QLACTLM2    0.715295
6   USEEQUIP    0.248342
7      BLIND    0.377324
8     DECIDE    0.659482
9   DIFFWALK    0.514461
10  DIFFDRES    0.264543
11  DIFFALON    0.431115
12  SMOKE100    0.758683
13  ADDEPEV2    0.457192
14   _PACAT1   12.667691

=== Resultados para grupo 77_88_99 ===
    Columna  Porcentaje
0  PHYSHLTH   64.525257
1  MENTHLTH   70.061720

=== Resultados para grupo 777_999 ===
    Columna  Porcentaje
0  EXEROFT1    0.960703

=== Resultados para grupo 9 ===
  Columna  Porcentaje
0   _RACE      1.6669

=== Resultados para grupo 14 ===
    Columna  Porcentaje
0  _AGEG5YR    1.194336


In [144]:
# De las variables preseleccionadas, se eliminan las que tienen mayor número de No Sabe/No contesto, Se nego (razonado en el documento: 1.2 Tercer cribado de columnas)
columnas_eliminar_3 = ['PHYSHLTH', 'MENTHLTH']

dataset_pretratado_3 = dataset_pretratado_2.drop(columns=columnas_eliminar_3, errors='ignore')

In [145]:
#Se comprueba que se tienen 22 variables y 1 target
dataset_pretratado_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 429360 entries, 0 to 441455
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   429358 non-null  float64
 1   BPHIGH4   429359 non-null  float64
 2   BPMEDS    172003 non-null  float64
 3   BLOODCHO  429360 non-null  float64
 4   HAVARTH3  429359 non-null  float64
 5   ADDEPEV2  429360 non-null  float64
 6   DIABETE3  429360 non-null  float64
 7   SEX       429360 non-null  float64
 8   QLACTLM2  420246 non-null  float64
 9   USEEQUIP  419180 non-null  float64
 10  BLIND     418473 non-null  float64
 11  DECIDE    417904 non-null  float64
 12  DIFFWALK  417330 non-null  float64
 13  DIFFDRES  416945 non-null  float64
 14  DIFFALON  416362 non-null  float64
 15  SMOKE100  415457 non-null  float64
 16  EXEROFT1  286457 non-null  float64
 17  _RACE     429360 non-null  float64
 18  _AGEG5YR  429360 non-null  float64
 19  _BMI5CAT  394205 non-null  float64
 20  _FRUTSUM 

In [146]:
# Se visualizan los NaN de las variables escogidas par ver como proceder
nulls = dataset_pretratado_3.isnull().sum().sort_values(ascending=False)
nulls_pct = (dataset_pretratado_3.isnull().sum()/len(dataset_pretratado_3)*100).sort_values(ascending=False)
pd.DataFrame({"Nulos": nulls, "Porcentaje": nulls_pct})

,Nulos,Porcentaje
BPMEDS,257357,59.939678
EXEROFT1,142903,33.282793
_VEGESUM,49754,11.587945
_FRUTSUM,42573,9.915456
_BMI5CAT,35155,8.187768
SMOKE100,13903,3.238075
DIFFALON,12998,3.027296
DIFFDRES,12415,2.891513
DIFFWALK,12030,2.801845
DECIDE,11456,2.668157


### Imputación de datos


In [147]:
# BPMEDS y su relación con BPHIGH (razonado en el documento: 2. Imputación de NaN (datos faltantes))

# Se crea una columna que indique si BPMEDS es nulo
dataset_pretratado_3['BPMEDS_nulos'] = dataset_pretratado_3['BPMEDS'].isna()

# Contar distribución de nulos según BPHIGH4
bphigh_distrib = pd.crosstab(dataset_pretratado_3['BPMEDS_nulos'], dataset_pretratado_3['BPHIGH4'], margins=True, normalize='columns')
print("Distribución de BPMEDS nulos según BPHIGH4:")
print(bphigh_distrib)

Distribución de BPMEDS nulos según BPHIGH4:
BPHIGH4       1.0  2.0  3.0  4.0  7.0  9.0       All
BPMEDS_nulos                                        
False         1.0  0.0  0.0  0.0  0.0  0.0  0.400604
True          0.0  1.0  1.0  1.0  1.0  1.0  0.599396


In [148]:
# Se observa que los NaNs es porque no tiene tensión alta, así que se sustituye por 2 (categoría de NO)
dataset_pretratado_3['BPMEDS'] = dataset_pretratado_3['BPMEDS'].fillna(2)

# Se verifica que no queden NaN en la columna
print(dataset_pretratado_3['BPMEDS'].isna().sum())

# Se elimina la columna creada de los nulos
dataset_pretratado_3.drop(columns='BPMEDS_nulos', inplace=True)

0


In [149]:
# EXEROFT1 se sigue el mismo procedimiento que en el caso anterior
dataset_pretratado_3['EXEROFT1_nulos'] = dataset_pretratado_3['EXEROFT1'].isna()

# Crosstab para ver relación de nulos en EXEROFT1 con movilidad limitada
for var in ['DIFFALON', 'DIFFDRES', 'DIFFWALK']:
    ct = pd.crosstab(dataset_pretratado_3['EXEROFT1_nulos'], dataset_pretratado_3[var], margins=True, normalize='columns')
    print(f"\nDistribución de nulos de EXEROFT1 según {var}:")
    print(ct)

# Se concluye que no tiene significado clínico, a diferencia del caso anterior (razonado en el documento: 2. Imputación de NaN (datos faltantes))
# Se elimina la columna creada
dataset_pretratado_3.drop(columns='EXEROFT1_nulos', inplace=True)


Distribución de nulos de EXEROFT1 según DIFFALON:
DIFFALON             1.0       2.0       7.0       9.0    All
EXEROFT1_nulos                                               
False           0.431847  0.711394  0.475485  0.180828  0.688
True            0.568153  0.288606  0.524515  0.819172  0.312

Distribución de nulos de EXEROFT1 según DIFFDRES:
DIFFDRES             1.0       2.0       7.0       9.0       All
EXEROFT1_nulos                                                  
False           0.398142  0.701415  0.436314  0.095368  0.687038
True            0.601858  0.298585  0.563686  0.904632  0.312962

Distribución de nulos de EXEROFT1 según DIFFWALK:
DIFFWALK             1.0       2.0       7.0       9.0       All
EXEROFT1_nulos                                                  
False           0.465478  0.734292  0.565015  0.177778  0.686392
True            0.534522  0.265708  0.434985  0.822222  0.313608


In [150]:
# Como en esta variable hay mas de 33% de valores nulos, se creará una nueva clase para ver, posteriormente como tratarla -> Pasara de float a object
dataset_pretratado_3['EXEROFT1'] = dataset_pretratado_3['EXEROFT1'].fillna('Desconocido')

In [151]:
dataset_pretratado_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 429360 entries, 0 to 441455
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   429358 non-null  float64
 1   BPHIGH4   429359 non-null  float64
 2   BPMEDS    429360 non-null  float64
 3   BLOODCHO  429360 non-null  float64
 4   HAVARTH3  429359 non-null  float64
 5   ADDEPEV2  429360 non-null  float64
 6   DIABETE3  429360 non-null  float64
 7   SEX       429360 non-null  float64
 8   QLACTLM2  420246 non-null  float64
 9   USEEQUIP  419180 non-null  float64
 10  BLIND     418473 non-null  float64
 11  DECIDE    417904 non-null  float64
 12  DIFFWALK  417330 non-null  float64
 13  DIFFDRES  416945 non-null  float64
 14  DIFFALON  416362 non-null  float64
 15  SMOKE100  415457 non-null  float64
 16  EXEROFT1  429360 non-null  object 
 17  _RACE     429360 non-null  float64
 18  _AGEG5YR  429360 non-null  float64
 19  _BMI5CAT  394205 non-null  float64
 20  _FRUTSUM 

## 4. Preparación para modelado

### 4.1. Dataset para: model_training_v1 - Sin imputar los NaN

In [152]:
# NOMBRE DEL DATASET: dataset_pretratado_3_sin_NaN
# Se eliminan los NaN faltantes para un primer modelo base (razonado en el documento: 2. Imputación de NaN (datos faltantes))
dataset_pretratado_3_sin_NaN = dataset_pretratado_3.copy()     # Se crea copia del dataset original
dataset_pretratado_3_sin_NaN = dataset_pretratado_3_sin_NaN.dropna()   # Se eliminan filas con NaN

# Se comprueba que no hay NaN en el dataset
dataset_pretratado_3_sin_NaN.info()

<class 'pandas.core.frame.DataFrame'>
Index: 350350 entries, 0 to 441455
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   350350 non-null  float64
 1   BPHIGH4   350350 non-null  float64
 2   BPMEDS    350350 non-null  float64
 3   BLOODCHO  350350 non-null  float64
 4   HAVARTH3  350350 non-null  float64
 5   ADDEPEV2  350350 non-null  float64
 6   DIABETE3  350350 non-null  float64
 7   SEX       350350 non-null  float64
 8   QLACTLM2  350350 non-null  float64
 9   USEEQUIP  350350 non-null  float64
 10  BLIND     350350 non-null  float64
 11  DECIDE    350350 non-null  float64
 12  DIFFWALK  350350 non-null  float64
 13  DIFFDRES  350350 non-null  float64
 14  DIFFALON  350350 non-null  float64
 15  SMOKE100  350350 non-null  float64
 16  EXEROFT1  350350 non-null  object 
 17  _RACE     350350 non-null  float64
 18  _AGEG5YR  350350 non-null  float64
 19  _BMI5CAT  350350 non-null  float64
 20  _FRUTSUM 

In [153]:
# Se comprueba cuantos NaN quedan en EXEROFT1 (Se creo una clase previamente con nombre 'Desconocido' que contenía todos sus NaN)
dataset_pretratado_3_sin_NaN['EXEROFT1'].value_counts().get('Desconocido')

np.int64(92641)

In [154]:
# Se tienen 350350 valores, aun restando 92641 de NaN de EXEFORT, quedarían 257709. Por lo que se elimina para esta primera prueba
dataset_pretratado_3_sin_NaN = dataset_pretratado_3_sin_NaN[dataset_pretratado_3_sin_NaN['EXEROFT1'] != 'Desconocido']

In [155]:
# Se comprueba que los números coinciden
dataset_pretratado_3_sin_NaN.info()

# Se vuelve a transformar EXEROFT1 a float
dataset_pretratado_3_sin_NaN['EXEROFT1'] = dataset_pretratado_3_sin_NaN['EXEROFT1'].astype(float)
print(dataset_pretratado_3_sin_NaN['EXEROFT1'].dtype)

<class 'pandas.core.frame.DataFrame'>
Index: 257709 entries, 1 to 441455
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   257709 non-null  float64
 1   BPHIGH4   257709 non-null  float64
 2   BPMEDS    257709 non-null  float64
 3   BLOODCHO  257709 non-null  float64
 4   HAVARTH3  257709 non-null  float64
 5   ADDEPEV2  257709 non-null  float64
 6   DIABETE3  257709 non-null  float64
 7   SEX       257709 non-null  float64
 8   QLACTLM2  257709 non-null  float64
 9   USEEQUIP  257709 non-null  float64
 10  BLIND     257709 non-null  float64
 11  DECIDE    257709 non-null  float64
 12  DIFFWALK  257709 non-null  float64
 13  DIFFDRES  257709 non-null  float64
 14  DIFFALON  257709 non-null  float64
 15  SMOKE100  257709 non-null  float64
 16  EXEROFT1  257709 non-null  object 
 17  _RACE     257709 non-null  float64
 18  _AGEG5YR  257709 non-null  float64
 19  _BMI5CAT  257709 non-null  float64
 20  _FRUTSUM 

In [156]:
# Se definen los valores especiales que indican "No sabe / Se negó". Hay 18, solo fatarían _BMI5CAT, _FRUTSUM_, VEGESUM, SEX y DIABETE3 que no presentan estas categorías
valores_invalidos = {
    'GENHLTH': [7, 9],
    'BPHIGH4': [7, 9],
    'BPMEDS': [7, 9],
    'BLOODCHO': [7, 9],
    'HAVARTH3': [7, 9],
    'QLACTLM2': [7, 9],
    'USEEQUIP': [7, 9],
    'BLIND': [7, 9],
    'DECIDE': [7, 9],
    'DIFFWALK': [7, 9],
    'DIFFDRES': [7, 9],
    'DIFFALON': [7, 9],
    'SMOKE100': [7, 9],
    'ADDEPEV2': [7, 9],
    '_PACAT1': [9],
    '_RACE': [9],
    '_AGEG5YR': [14],
    'EXEROFT1': [777, 999]
}

# Se reemplazan estos valores por NaN para posteriormente sustituirlos por las categorías en función del tipo de variable
for col, vals in valores_invalidos.items():
    dataset_pretratado_3_sin_NaN[col] = dataset_pretratado_3_sin_NaN[col].replace(vals, np.nan)

In [157]:
# Separar tipos de variables
categorical_nominal = ['BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_RACE', 'SEX']
categorical_ordinal = ['GENHLTH', '_PACAT1', '_AGEG5YR', '_BMI5CAT']
numerical = ['EXEROFT1', '_FRUTSUM', '_VEGESUM']

print(f'Comprobación total:{len(categorical_nominal)+len(categorical_ordinal)+len(numerical)}')

Comprobación total:22


In [158]:
# Se imputan
# Categóricas nominales. Se imputan con número para mantener el tipo de dato consistente (todos son float)
for col in categorical_nominal:
    dataset_pretratado_3_sin_NaN[col] = dataset_pretratado_3_sin_NaN[col].fillna(-1)

# Categóricas ordinales. Se imputan con número para mantener el tipo de dato consistente (todos son float)
for col in categorical_ordinal:
    dataset_pretratado_3_sin_NaN[col] = dataset_pretratado_3_sin_NaN[col].fillna(-1)

# Numéricas. Se usará la mediana
for col in numerical:
    dataset_pretratado_3_sin_NaN[col] = dataset_pretratado_3_sin_NaN[col].fillna(dataset_pretratado_3_sin_NaN[col].median())

In [159]:
# Se reindexa el dataset
dataset_pretratado_3_sin_NaN = dataset_pretratado_3_sin_NaN.reset_index(drop=True)

In [160]:
# Se comprueba que está todo correcto antes de exportar
dataset_pretratado_3_sin_NaN.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257709 entries, 0 to 257708
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   257709 non-null  float64
 1   BPHIGH4   257709 non-null  float64
 2   BPMEDS    257709 non-null  float64
 3   BLOODCHO  257709 non-null  float64
 4   HAVARTH3  257709 non-null  float64
 5   ADDEPEV2  257709 non-null  float64
 6   DIABETE3  257709 non-null  float64
 7   SEX       257709 non-null  float64
 8   QLACTLM2  257709 non-null  float64
 9   USEEQUIP  257709 non-null  float64
 10  BLIND     257709 non-null  float64
 11  DECIDE    257709 non-null  float64
 12  DIFFWALK  257709 non-null  float64
 13  DIFFDRES  257709 non-null  float64
 14  DIFFALON  257709 non-null  float64
 15  SMOKE100  257709 non-null  float64
 16  EXEROFT1  257709 non-null  float64
 17  _RACE     257709 non-null  float64
 18  _AGEG5YR  257709 non-null  float64
 19  _BMI5CAT  257709 non-null  float64
 20  _FRU

In [161]:
# Se exportan dataset limpio
dataset_pretratado_3_sin_NaN.to_csv("/content/drive/MyDrive/10ª Edición Máster en Data Science/TFM/clean_data_v1.csv", index=False)